In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## IMPORT

In [2]:
import pandas as pd
import numpy as np
import torch
BASE_PATH = "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1"
IMG_PATH = BASE_PATH + "/images/"

## DATASET LOADING

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/test.csv")

train_df = train_df.sample(15000, random_state=42)
print(train_df.shape)
print(train_df.columns)
train_df.head()

(15000, 21)
Index(['id', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
       'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass',
       'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax',
       'Pneumoperitoneum', 'Pneumomediastinum', 'Subcutaneous Emphysema',
       'Tortuous Aorta', 'Calcification of the Aorta', 'No Finding'],
      dtype='object')


,id,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,...,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,Pneumoperitoneum,Pneumomediastinum,Subcutaneous Emphysema,Tortuous Aorta,Calcification of the Aorta,No Finding
12757,8ae9858c14b8491dbf15012d184e68fb.png,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
41363,7c4b16f3a66b44a5998df7beaa96ff3e.png,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
8781,eb6fbb5330424489af0d606029d1d188.png,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
44019,d008cd3905864631a4effc7e412b34b6.png,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
43365,7d48f0ec570f4b8997e34754b158560b.png,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


## TRAIN-TEST SPLIT

In [4]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df.iloc[:,1:].values.argmax(axis=1)
)

## CREATING DATASET CLASS

In [5]:
import torch 
from torch.utils.data import Dataset
from PIL import Image
import os

IMG_PATH = "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/images/"

class XrayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]

        img_name = row['id']
        img_path = os.path.join(IMG_PATH, img_name)

        image = Image.open(img_path).convert("RGB")

        label = row[1:].values.argmax() # converting one_hot to class idx

        if self.transform:
            image = self.transform(image)

        return image, label
        

## TRANSFORM AND DATALOADER

In [6]:
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

train_dataset = XrayDataset(train_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, num_workers=2, shuffle=True)

val_dataset = XrayDataset(val_data, transform=test_transform)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

## CHECKING DATALOADER

In [7]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([16, 3, 224, 224]) torch.Size([16])


## FOCAL LOSS

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
        pt = torch.exp(-ce_loss)
        loss = ((1-pt) ** self.gamma * ce_loss).mean()
        return loss

## MODEL

In [9]:
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet50(weights=ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 20)

model = model.to(device)
criterion = FocalLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 244MB/s]


## TRAINING AND VALIDATION LOOP

In [10]:
best_loss = float('inf')

for epoch in range(3):
    model.train()
    total_loss = 0

    for images, lables in train_loader:
        images = images.to(device) # changing to appropriate device
        labels = lables.to(device) # changing to appropriate device
    
        outputs = model(images) # prediction
        loss = criterion(outputs,labels) # loss
        optimizer.zero_grad() # make gradient 0 , so no accumulation of gradients
        loss.backward() # backpropagate using chain rule
        optimizer.step() # update the parmeters
    
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss:{total_loss/len(train_loader)}")

    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
    
            outputs = model(images)
            loss = criterion(outputs, labels)
    
            val_loss += loss.item()
    
    print(f"Val Loss: {val_loss/len(val_loader)}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1, Loss:0.9578043662154598
Val Loss: 0.8288909255507144
Epoch 2, Loss:0.8381364340546416
Val Loss: 0.7413658665374239
Epoch 3, Loss:0.7591967708997126
Val Loss: 0.6911975831744519


## TEST DATASET

In [11]:
class TestDataset(Dataset):
    def __init__(self,df,transform=None):
        self.df = df
        self.transform = test_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        img_name = self.df.iloc[idx]['id']
        img_path = os.path.join(IMG_PATH,img_name)
        
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, img_name

##  TEST DATASET AND LOADER

In [12]:
test_dataset = TestDataset(test_df, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

## PREDICITON

In [13]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

predictions = []

with torch.no_grad():
    for images, img_names in test_loader:
        images = images.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.cpu().numpy())

## SUBMISSION

In [14]:
import numpy as np

num_classes = 20
one_hot = np.zeros((len(predictions), num_classes))

for i, p in enumerate(predictions):
    one_hot[i, p] = 1

In [15]:
submission = pd.DataFrame(one_hot, columns=train_df.columns[1:])
submission.insert(0, 'id', test_df['id'])
print(submission.head())
submission.to_csv("submission.csv", index=False)

                                     id  Atelectasis  Cardiomegaly  \
0  7b647fbfcc874a7084a4470fc150e267.png          0.0           0.0   
1  cc804b94d80c4a80a206298c307adfec.png          0.0           0.0   
2  1df09c3becd04de995244caae36ddf57.png          0.0           0.0   
3  044cac47cfdf4c8b90848c9e56c36bfa.png          0.0           0.0   
4  a873523c43664a049c5e8f26add7ecb2.png          0.0           0.0   

   Consolidation  Edema  Effusion  Emphysema  Fibrosis  Hernia  Infiltration  \
0            0.0    0.0       0.0        0.0       0.0     0.0           0.0   
1            0.0    0.0       0.0        0.0       0.0     0.0           0.0   
2            0.0    0.0       0.0        0.0       0.0     0.0           0.0   
3            0.0    0.0       0.0        0.0       0.0     0.0           0.0   
4            0.0    0.0       0.0        0.0       0.0     0.0           0.0   

   ...  Nodule  Pleural_Thickening  Pneumonia  Pneumothorax  Pneumoperitoneum  \
0  ...     0.0   